In [1]:
import requests
from datetime import datetime
import random

In [2]:
url = "https://api.jolpi.ca/ergast/f1/drivers.json?limit=1000"

respuesta = requests.get(url)
respuesta.raise_for_status()

datos = respuesta.json()

pilotos = datos["MRData"]["DriverTable"]["Drivers"]

print("Cantidad de pilotos:", len(pilotos))

Cantidad de pilotos: 100


In [3]:
pilotos_por_mes = {}

for piloto in pilotos:

    fecha = piloto.get("dateOfBirth")

    if fecha:
        mes = datetime.strptime(
            fecha,
            "%Y-%m-%d"
        ).month

        if mes not in pilotos_por_mes:
            pilotos_por_mes[mes] = []

        pilotos_por_mes[mes].append(piloto)

In [4]:
piloto = random.choice(
    pilotos_por_mes[mes]
)

In [5]:
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

In [7]:
cache_estadisticas = {}

def obtener_estadisticas_cache(piloto):

    driver_id = piloto["driverId"]

    if driver_id not in cache_estadisticas:
        cache_estadisticas[driver_id] = obtener_estadisticas(piloto)

    return cache_estadisticas[driver_id]

obtener_estadisticas_cache(piloto)

{'victorias': 0,
 'podios': 0,
 'puntos': 0.0,
 'carreras': 1,
 'mayor_remontada': 7,
 'carrera_remontada': 'United States Grand Prix',
 'posicion_inicial': 25,
 'posicion_final': 18}

In [8]:
def pilotos_con_minimo_de_carreras(mes, minimo=20):

    candidatos = pilotos_por_mes.get(mes, [])

    candidatos_validos = []

    for piloto in candidatos:

        estadisticas = obtener_estadisticas_cache(piloto)

        if estadisticas["carreras"] >= minimo:
            candidatos_validos.append(piloto)

    return candidatos_validos

In [11]:
def obtener_dato_curioso(estadisticas):

    remontada = estadisticas["mayor_remontada"]

    if remontada > 0:
        return (
            f"Su mayor remontada fue de {remontada} posiciones: "
            f"largó {estadisticas['posicion_inicial']}° "
            f"y terminó {estadisticas['posicion_final']}° "
            f"en {estadisticas['carrera_remontada']}."
        )

    return "No encontramos una remontada destacada."

In [14]:
cache_estadisticas = {}
pilotos_20 = []
pilotos_por_mes_20 = {}

In [41]:
resultados_pilotos = {}

for temporada in range(2010, 2027):

    url = f"https://api.jolpi.ca/ergast/f1/{temporada}/results.json?limit=100"

    respuesta = requests.get(url)
    respuesta.raise_for_status()

    datos = respuesta.json()

    carreras = datos["MRData"]["RaceTable"]["Races"]

    for carrera in carreras:

        for resultado in carrera.get("Results", []):

            driver_id = resultado["Driver"]["driverId"]

            if driver_id not in resultados_pilotos:
                resultados_pilotos[driver_id] = {
                    "piloto": resultado["Driver"],
                    "carreras": 0
                }

            resultados_pilotos[driver_id]["carreras"] += 1

print("Pilotos encontrados:", len(resultados_pilotos))

Pilotos encontrados: 78


In [44]:
pilotos_10 = []

for driver_id, datos_piloto in resultados_pilotos.items():

    if datos_piloto["carreras"] >= 10:
        pilotos_10.append(datos_piloto["piloto"])

print("Pilotos con 10+ carreras:", len(pilotos_10))

Pilotos con 10+ carreras: 46


In [47]:
pilotos_por_mes_10 = {}

for piloto in pilotos_10:

    fecha = piloto.get("dateOfBirth")

    if fecha:

        mes = datetime.strptime(
            fecha,
            "%Y-%m-%d"
        ).month

        if mes not in pilotos_por_mes_10:
            pilotos_por_mes_10[mes] = []

        pilotos_por_mes_10[mes].append(piloto)

In [48]:
for mes in range(1, 13):

    candidatos = pilotos_por_mes_10.get(mes, [])

    print(mes, "→", len(candidatos), "pilotos")

1 → 5 pilotos
2 → 3 pilotos
3 → 5 pilotos
4 → 6 pilotos
5 → 3 pilotos
6 → 2 pilotos
7 → 3 pilotos
8 → 5 pilotos
9 → 7 pilotos
10 → 5 pilotos
11 → 1 pilotos
12 → 1 pilotos


In [51]:
def obtener_estadisticas(piloto):

    driver_id = piloto["driverId"]

    url = f"https://api.jolpi.ca/ergast/f1/drivers/{driver_id}/results.json?limit=100"

    respuesta = requests.get(url)
    respuesta.raise_for_status()

    datos = respuesta.json()

    carreras = datos["MRData"]["RaceTable"]["Races"]

    victorias = 0
    podios = 0
    puntos = 0
    carreras_disputadas = 0

    mayor_remontada = 0
    carrera_remontada = ""
    posicion_inicial = 0
    posicion_final = 0

    for carrera in carreras:

        resultados = carrera.get("Results", [])

        for resultado in resultados:

            carreras_disputadas += 1

            posicion = resultado.get("position")
            grid = resultado.get("grid")

            if posicion == "1":
                victorias += 1

            if posicion in ["1", "2", "3"]:
                podios += 1

            puntos += float(resultado.get("points", 0))

            # Buscar mayor remontada
            if grid and posicion:

                try:
                    grid_num = int(grid)
                    posicion_num = int(posicion)

                    remontada = grid_num - posicion_num

                    if remontada > mayor_remontada:

                        mayor_remontada = remontada
                        carrera_remontada = carrera["raceName"]
                        posicion_inicial = grid_num
                        posicion_final = posicion_num

                except ValueError:
                    pass

    return {
        "victorias": victorias,
        "podios": podios,
        "puntos": puntos,
        "carreras": carreras_disputadas,
        "mayor_remontada": mayor_remontada,
        "carrera_remontada": carrera_remontada,
        "posicion_inicial": posicion_inicial,
        "posicion_final": posicion_final
    }

In [52]:
def que_piloto_soy(mes):

    if mes < 1 or mes > 12:
        return "El mes debe estar entre 1 y 12."

    candidatos = pilotos_por_mes_10.get(mes, [])

    if not candidatos:
        return "No encontramos pilotos con al menos 10 carreras en ese mes."

    piloto = random.choice(candidatos)

    nombre = piloto["givenName"]
    apellido = piloto["familyName"]
    fecha = piloto["dateOfBirth"]
    nacionalidad = piloto.get("nationality", "Desconocida")

    estadisticas = obtener_estadisticas(piloto)

    return (
        f"🏎️ TU PILOTO ES {nombre.upper()} {apellido.upper()}!\n\n"
        f"🌍 Nacionalidad: {nacionalidad}\n"
        f"📅 Fecha de nacimiento: {fecha}\n\n"
        f"🏆 Victorias: {estadisticas['victorias']}\n"
        f"🥇 Podios: {estadisticas['podios']}\n"
        f"🏁 Carreras: {estadisticas['carreras']}\n"
        f"⭐ Puntos: {estadisticas['puntos']:.1f}"
    )

In [53]:
mes = input("¿Cuál es tu mes de nacimiento? ").lower()

if mes in meses:
    print(que_piloto_soy(meses[mes]))
else:
    print("No reconocí ese mes.")

🏎️ TU PILOTO ES ANTONIO GIOVINAZZI!

🌍 Nacionalidad: Italian
📅 Fecha de nacimiento: 1993-12-14

🏆 Victorias: 0
🥇 Podios: 0
🏁 Carreras: 62
⭐ Puntos: 21.0
